# Create Keio Medical Science Prize Awards from Official Laureate Listing

Creates OpenAlex award rows from the Keio University Medical Science Fund official Keio Medical Science Prize laureate pages.

**Prerequisites:**
- Admin/human with AWS credentials runs `scripts/local/keio_medical_science_prize_to_s3.py` to fetch the official HTML pages, write parquet, and upload it.
- Contractor has prepared the script, notebook, registry entry, tracker row, and local QA, but does not have S3/Databricks access.
- The downloader writes parquet columns as strings per `plans/awards/how-to-add-a-funder.md`; this notebook does type conversion with `TRY_CAST` / `TRY_TO_DATE`.

**Data sources:**
- Current prize page: `https://www.ms-fund.keio.ac.jp/en/prize/`
- Past laureates table: `https://www.ms-fund.keio.ac.jp/en/prize/list.html`
- Detail pages for 1999 onward, e.g. `https://www.ms-fund.keio.ac.jp/en/prize/the-2025-keio-medical-science-prize-awardees.html`
- Amount source: official 2026 nomination PDF at `https://www.ms-fund.keio.ac.jp/en/news/a4246b3e370ad2125f9f720963fda4963f374cad.pdf`

**S3 location:** `s3a://openalex-ingest/awards/keio_medical_science_prize/keio_medical_science_prize_laureates.parquet`

**Local full-source validation on 2026-05-26:** 57 official laureate rows, 1996-2025. Coverage: 100% title/year/landing-url/amount/currency; 51/57 descriptions and affiliation strings (89.5%) from official detail pages. The 1996-1998 rows are present on the official roster but have no official detail pages, so description and affiliation remain NULL by source authority.

**OpenAlex funder mapping:**
- Chosen funder_id: 4320320909
- Display name: Keio University
- Country: JP
- DOI/ROR: not present on the OpenAlex funder row
- Rationale: the official Keio University Medical Science Fund pages state that Keio University annually awards the prize and that award events are held at Keio University.

**Mapping summary:** One OpenAlex award row per official laureate. `amount=10000000` and `currency='JPY'` are populated for all rows from the official nomination PDF's statement that each laureate receives a monetary award of 10 million yen. `lead_investigator` carries the laureate name and official affiliation/position text when available; co-lead and investigators arrays are NULL because this is one row per individual laureate.

## Step 1: Load Raw Parquet

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.keio_medical_science_prize_raw
USING delta
AS
SELECT
    *,
    current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/keio_medical_science_prize/keio_medical_science_prize_laureates.parquet`;

In [ ]:
%sql
-- Check raw row count. Local validation on 2026-05-26 found 57 rows.
SELECT COUNT(*) as total_keio_laureates
FROM openalex.awards.keio_medical_science_prize_raw;

In [ ]:
%sql
-- Verify actual column names before transform logic references them.
DESCRIBE openalex.awards.keio_medical_science_prize_raw;

In [ ]:
%sql
-- Sample raw Keio data.
SELECT
    funder_award_id,
    display_name,
    source_year,
    laureate_name,
    affiliation_raw,
    amount,
    currency,
    landing_page_url
FROM openalex.awards.keio_medical_science_prize_raw
ORDER BY source_year DESC, laureate_position
LIMIT 10;

## Step 1.5: Source, Money, and Key Scans

In [ ]:
%sql
-- Money-field scan per runbook Step 1.5. Keio amount is populated from the official nomination PDF.
SELECT column_name
FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'keio_medical_science_prize_raw'
  AND LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|funding|cost|budget|grant|award|currency';

In [ ]:
%sql
-- Confirm amount/date/source coverage before mapping.
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS rows_with_title,
    COUNT(source_year) AS rows_with_year,
    COUNT(laureate_name) AS rows_with_laureate,
    COUNT(description) AS rows_with_description,
    COUNT(affiliation_raw) AS rows_with_affiliation,
    COUNT(amount) AS rows_with_amount,
    COUNT(currency) AS rows_with_currency,
    MIN(TRY_CAST(source_year AS INT)) AS min_year,
    MAX(TRY_CAST(source_year AS INT)) AS max_year,
    SUM(TRY_CAST(amount AS DOUBLE)) AS total_jpy
FROM openalex.awards.keio_medical_science_prize_raw;

In [ ]:
%sql
-- Native-key inspection.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(DISTINCT CONCAT(source_year, ':', laureate_position, ':', laureate_name)) AS distinct_source_positions
FROM openalex.awards.keio_medical_science_prize_raw;

In [ ]:
%sql
-- Detail coverage by year; 1996-1998 are roster-only on the official site.
SELECT
    source_year,
    COUNT(*) AS rows,
    COUNT(description) AS with_description,
    COUNT(affiliation_raw) AS with_affiliation
FROM openalex.awards.keio_medical_science_prize_raw
GROUP BY source_year
ORDER BY source_year;

In [ ]:
%sql
-- Amount/currency distribution. Expected: all rows JPY 10,000,000.
SELECT
    currency,
    TRY_CAST(amount AS DOUBLE) AS amount,
    COUNT(*) AS rows
FROM openalex.awards.keio_medical_science_prize_raw
GROUP BY currency, TRY_CAST(amount AS DOUBLE)
ORDER BY rows DESC;

## Step 1.6: Funder Existence Fail-Fast

In [ ]:
%sql
-- Must return TRUE. If this fails, STOP: Keio University is missing from openalex.common.funder.
SELECT assert_true(
    COUNT(*) = 1,
    'Expected exactly one openalex.common.funder row for Keio University F4320320909'
) AS funder_row_exists
FROM openalex.common.funder
WHERE funder_id = 4320320909;

In [ ]:
%sql
SELECT funder_id, display_name, country_code, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320320909;

## Step 2: Transform to OpenAlex Awards Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.keio_medical_science_prize_awards
USING delta
AS
WITH
keio_funder AS (
    SELECT
        funder_id,
        display_name,
        ror_id,
        doi
    FROM openalex.common.funder
    WHERE funder_id = 4320320909
),

raw_prepared AS (
    SELECT
        *,
        LOWER(TRIM(funder_award_id)) AS native_award_id,
        TRY_CAST(amount AS DOUBLE) AS parsed_amount,
        TRY_TO_DATE(start_date, 'yyyy-MM-dd') AS parsed_start_date,
        TRY_TO_DATE(end_date, 'yyyy-MM-dd') AS parsed_end_date,
        TRY_CAST(source_year AS INT) AS parsed_year
    FROM openalex.awards.keio_medical_science_prize_raw
    WHERE funder_award_id IS NOT NULL
      AND TRIM(funder_award_id) != ''
      AND display_name IS NOT NULL
      AND TRIM(display_name) != ''
),

awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000 as id,

        TRIM(g.display_name) as display_name,

        CASE
            WHEN g.description IS NULL OR TRIM(g.description) = '' THEN NULL
            ELSE TRIM(g.description)
        END as description,

        f.funder_id,
        g.native_award_id as funder_award_id,

        g.parsed_amount as amount,
        NULLIF(TRIM(g.currency), '') as currency,

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,

        'prize' as funding_type,

        COALESCE(NULLIF(TRIM(g.funder_scheme), ''), 'Keio Medical Science Prize') as funder_scheme,

        'keio_medical_science_prize' as provenance,

        g.parsed_start_date as start_date,
        g.parsed_end_date as end_date,
        COALESCE(YEAR(g.parsed_start_date), g.parsed_year) as start_year,
        COALESCE(YEAR(g.parsed_end_date), g.parsed_year) as end_year,

        struct(
            NULLIF(TRIM(g.given_name), '') as given_name,
            NULLIF(TRIM(g.family_name), '') as family_name,
            CAST(NULL AS STRING) as orcid,
            g.parsed_start_date as role_start,
            struct(
                NULLIF(TRIM(g.affiliation_raw), '') as name,
                CAST(NULL AS STRING) as country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
            ) as affiliation
        ) as lead_investigator,

        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,

        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,

        NULLIF(TRIM(g.landing_page_url), '') as landing_page_url,
        CAST(NULL AS STRING) as doi,

        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000) as works_api_url,

        current_timestamp() as created_date,
        current_timestamp() as updated_date

    FROM raw_prepared g
    CROSS JOIN keio_funder f
)

SELECT * FROM awards_transformed;

## Step 3: Delete Previous Source Rows and Insert Priority 125

In [ ]:
%sql
-- Remove previous Keio Medical Science Prize rows before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'keio_medical_science_prize' AND priority = 125;

-- Insert into openalex_awards_raw with exact OpenAlex awards schema plus priority.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    125 as priority
FROM openalex.awards.keio_medical_science_prize_awards;

## Verification

In [ ]:
%sql
SELECT COUNT(*) as total_keio_awards
FROM openalex.awards.keio_medical_science_prize_awards;

In [ ]:
%sql
-- Confirm the transformed table has the OpenAlex awards columns only.
DESCRIBE openalex.awards.keio_medical_science_prize_awards;

In [ ]:
%sql
-- Sample transformed awards.
SELECT
    id,
    display_name,
    funder_award_id,
    funder_scheme,
    funding_type,
    amount,
    currency,
    start_year,
    lead_investigator.given_name AS given_name,
    lead_investigator.family_name AS family_name,
    lead_investigator.affiliation.name AS affiliation,
    landing_page_url
FROM openalex.awards.keio_medical_science_prize_awards
ORDER BY start_year DESC, funder_award_id
LIMIT 10;

In [ ]:
%sql
-- Check ID and native-key uniqueness.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT id) AS distinct_openalex_ids,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids
FROM openalex.awards.keio_medical_science_prize_awards;

In [ ]:
%sql
-- Check funder distribution. Should be only F4320320909.
SELECT funder.display_name, funder_id, COUNT(*) as cnt
FROM openalex.awards.keio_medical_science_prize_awards
GROUP BY funder.display_name, funder_id
ORDER BY cnt DESC;

In [ ]:
%sql
-- Data completeness.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(currency) as has_currency,
    COUNT(start_date) as has_start_date,
    COUNT(end_date) as has_end_date,
    COUNT(landing_page_url) as has_url,
    COUNT(lead_investigator) as has_lead,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) as pct_description,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) as pct_amount
FROM openalex.awards.keio_medical_science_prize_awards;

In [ ]:
%sql
-- Amount/currency verification. Expected: 57 rows at JPY 10,000,000.
SELECT currency, amount, COUNT(*) AS rows
FROM openalex.awards.keio_medical_science_prize_awards
GROUP BY currency, amount
ORDER BY rows DESC;

In [ ]:
%sql
-- Year distribution.
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.keio_medical_science_prize_awards
GROUP BY start_year
ORDER BY start_year DESC;

In [ ]:
%sql
-- Verify rows inserted into the raw awards table at priority 125.
SELECT
    priority,
    provenance,
    COUNT(*) as cnt,
    COUNT(DISTINCT funder_award_id) as distinct_funder_award_ids
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'keio_medical_science_prize' AND priority = 125
GROUP BY priority, provenance;